# Chapter 5: And Then There Were Many

This notebook accompanies **Chapter 5** of the lecture notes.

**Agenda**

☕ · 🧭 · 🎲 · 🦘 · 📊 · 🏁

**Next steps (take it from here):** 🗺️ · 🔧

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.


*Welcome to the Coffee Flavor Lab.* Today we search for the perfect brew. Imagine a landscape of coffee flavor  -  each hill and valley corresponds to a combination of brewing parameters (grind size, water temperature, extraction time). Some parameter settings produce a decent cup; one particular setting produces the **best cup**. Our optimizers are taste-testers navigating this landscape.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import check_random_restart, check_basin_hopping

## The Landscape

Newton's method finds a stationary point near where you start. When a function has multiple local minima, the one Newton converges to depends entirely on the initial guess. This chapter explores strategies for escaping local traps and searching for the **global** minimum.

We use a 1D Shekel function  -  a sum of attractive wells ("foxholes")  -  as our test landscape. Think of each foxhole as a **local optimum in brewing parameter space**: a set of brewing parameters that tastes pretty good, but maybe not the best. The deepest well is the best-tasting coffee. Our job is to find it.

The code below is a simplified version you can use throughout the notebook.

### Shekel Function (provided)

In [ ]:
class Shekel:
    """
    1D Shekel function: f(x) = -sum_i { c_i / (r_i + (x - x_i)^2) }

    Parameters
    positions : list  -  hole centres (x_i)
    c_values  : list  -  depth coefficients (larger = deeper)
    r_values  : list  -  width parameters (larger = wider)
    """

    def __init__(self, positions, c_values=None, r_values=None):
        self.positions = np.asarray(positions, dtype=float)
        self.n_holes = len(self.positions)
        self.c_values = np.ones(self.n_holes) if c_values is None else np.asarray(c_values, dtype=float)
        self.r_values = np.ones(self.n_holes) if r_values is None else np.asarray(r_values, dtype=float)

    def __call__(self, x):
        result = 0.0
        for i in range(self.n_holes):
            result -= self.c_values[i] / (self.r_values[i] + (x - self.positions[i]) ** 2)
        return result

    def derivative(self, x):
        result = 0.0
        for i in range(self.n_holes):
            denom = self.r_values[i] + (x - self.positions[i]) ** 2
            result += 2 * self.c_values[i] * (x - self.positions[i]) / (denom ** 2)
        return result

    def second_derivative(self, x):
        result = 0.0
        for i in range(self.n_holes):
            dx = x - self.positions[i]
            r = self.r_values[i]
            denom = r + dx ** 2
            numerator = 2 * r - 6 * dx ** 2
            result -= self.c_values[i] * numerator / (denom ** 3)
        return result

    def plot(self, bounds=(-2, 12), n_plot=500):
        xs = np.linspace(bounds[0], bounds[1], n_plot)
        ys = np.array([self(xi) for xi in xs])
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(xs, ys, color=_GOLDEN, ls='-', linewidth=1.2)
        for i in range(self.n_holes):
            ax.axvline(self.positions[i], color=_BORDER, ls=':', lw=0.8)
        ax.set_xlabel('x')
        ax.set_ylabel('f(x)')
        tufte_axis(ax)
        plt.tight_layout()
        plt.show()

### Newton's Method for Optimization (provided)

This is a silent version of Newton's method that finds a stationary point of f by solving f'(x) = 0. It returns the converged x, the full history of iterates, and the number of derivative evaluations.

In [ ]:
def newton_optimize(f, f_prime, f_double_prime, x0, tol=1e-8, max_iter=100):
    """
    Newton's method applied to f'(x) = 0.

    Returns
    x_star  : float  -  approximate stationary point
    history : list   -  all x-values visited [x0, x1, ...]
    n_evals : int    -  derivative evaluation count
    """
    x = x0
    history = [x]
    n_evals = 1
    for i in range(max_iter):
        fpx = f_prime(x)
        fppx = f_double_prime(x)
        if abs(fppx) < 1e-15:
            break
        x_new = x - fpx / fppx
        history.append(x_new)
        n_evals += 2
        if abs(x_new - x) < tol:
            break
        x = x_new
    return history[-1], history, n_evals

### Build the Landscape

Three foxholes at positions 1, 5, and 9  -  think of them as three local sweet spots in our brewing parameter space. The well at x=5 is the deepest (c=3) and widest (r=0.5)  -  that is the **best-tasting coffee**, our global minimum. The other two are decent brews but not the best.

In [ ]:
landscape = Shekel(
    positions=[1, 5, 9],
    c_values=[0.5, 3, 1],
    r_values=[0.3, 0.5, 0.2],
)

f = landscape
f_prime = landscape.derivative
f_double_prime = landscape.second_derivative

BOUNDS = (-2, 12)

landscape.plot(bounds=BOUNDS)

## Exercises

### ☕ Explore the Landscape

> The flavor landscape has three wells of different depths and widths  -  three local sweet spots in coffee parameter space. Before running any optimizer, look at the plot and estimate: where is the best brew (global minimum), and how much better is it compared to the runners-up?

<details><summary>Thought</summary>

The well at x=5 has the largest depth coefficient (c=3) and a moderate width (r=0.5). At its centre, f(5) = -0.5/16.3 - 3/0.5 - 1/16.2, which is approximately -6.09. The wells at x=1 and x=9 are much shallower: f(1) is roughly -1.73 and f(9) about -5.06. The global minimum  -  the best coffee  -  is clearly at x=5.
</details>

Evaluate the function at each hole centre and at a few points between holes to build intuition.

In [ ]:
# Evaluate at hole centres
for pos in [1, 5, 9]:
    print(f'f({pos}) = {f(pos):.6f}')

print()

# Evaluate between holes
for x_val in [0, 3, 7, 11]:
    print(f'f({x_val}) = {f(x_val):.6f}')

### 🧭 Newton from Different Starting Points

> Imagine you are a barista who can only taste-test nearby recipes and hill-climb to the nearest good brew. If you start tasting at x=3 (between the first and second sweet spots), which local optimum will you converge to? What about x=7? Newton finds the nearest good cup  -  but is it THE best cup?

<details><summary>Thought</summary>

At x=3 the slope is negative (pointing towards x=5, the deeper well). Newton follows the curvature and converges to x=5  -  the best coffee. At x=7 the slope depends on the relative pull of the wells at x=5 and x=9. The well at x=5 is much deeper, so its gradient influence extends further. You need to try it to see.
</details>

Run Newton from several starting points and observe which local minimum each converges to. Try x0 values of 0, 3, 7, and 11.

In [ ]:
starts = [0, 3, 7, 11]

fig, ax = plt.subplots(figsize=(10, 4))
xs_plot = np.linspace(BOUNDS[0], BOUNDS[1], 500)
ys_plot = np.array([f(xi) for xi in xs_plot])
ax.plot(xs_plot, ys_plot, color=_GOLDEN, ls='-', linewidth=1.2)

colors = [_ACCENT, _ORANGE, _TERRA, _ACCENT]

for x0, color in zip(starts, colors):
    x_star, history, n_evals = newton_optimize(f, f_prime, f_double_prime, x0)
    ax.plot(x0, f(x0), 'o', color=color, markersize=8, markeredgecolor=_BORDER, zorder=4)
    ax.plot(x_star, f(x_star), '*', color=color, markersize=14, markeredgecolor=_BORDER, zorder=5)
    ax.annotate('', xy=(x_star, f(x_star)), xytext=(x0, f(x0)),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2,
                                connectionstyle='arc3,rad=0.2'))
    print(f'x0={x0:5.1f}  \u2192  x*={x_star:8.5f}  f(x*)={f(x_star):.6f}  '
          f'({len(history)-1} iters, {n_evals} evals)')

tufte_axis(ax)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
plt.tight_layout()
plt.show()

**Observe:**
- Newton always converges to a **local** minimum near the starting point  -  the nearest good brew, not necessarily the best one.
- It never jumps over a hill to reach a deeper well.
- Starting from x=0 and x=3 both converge to the same well. The basin of attraction of the best coffee at x=5 is wide.
- To find the **best cup** reliably, we need a strategy that explores multiple basins  -  like sending multiple baristas to taste-test from different starting recipes.

### 🎲 Random Restart

> If each Newton run (each barista starting from a random recipe) has a probability p of landing in the basin of the best coffee, how many independent restarts K do you need so that the probability of finding the best brew at least once exceeds 95%? Recall that P(find global) = 1 - (1-p)^K.

<details><summary>Thought</summary>

For this landscape, the basin of the best coffee at x=5 covers roughly 40-50% of the search domain, so p is around 0.4-0.5. With p=0.4, we need K such that 1 - 0.6^K > 0.95, which gives K > log(0.05)/log(0.6) = 5.9, so about 6 restarts. With 10 restarts we are very likely to find the best brew.
</details>

Implement `random_restart`. It should draw `n_restarts` starting points uniformly from `bounds` (each one a random recipe), run Newton from each, and track the best result found.

Return `(best_x, best_f, all_results)` where `all_results` is a list with one entry per restart (a dict or tuple with at least the start, converged x, and f-value).

Useful operations: `np.random.uniform(low, high)`.

In [ ]:
def random_restart(newton_fn, f, f_prime, f_double_prime, n_restarts, bounds):
    """Run Newton from n_restarts random starting points, return the best."""
    # Step 1: initialise best_x = None, best_f = float('inf'), all_results = []

    # Step 2: loop n_restarts times:
    #   - sample x0 = np.random.uniform(bounds[0], bounds[1])
    #   - call newton_fn(f, f_prime, f_double_prime, x0) -> (x_star, history, n_evals)
    #   - evaluate f_star = f(x_star)
    #   - append {'x0': x0, 'x_star': x_star, 'f_star': f_star} to all_results
    #   - update best_x, best_f if f_star < best_f

    # Step 3: return (best_x, best_f, all_results)


check_random_restart(
    random_restart, newton_optimize,
    f, f_prime, f_double_prime,
    n_restarts=10, bounds=BOUNDS,
    known_global_x=5.0, known_global_f=-6.09,
)

### 🦘 Basin Hopping

> Random restart sends each barista to a completely random recipe  -  no memory of where good brews were found. Basin hopping starts from the current best recipe and makes a random jump, then converges again. It is like a barista who remembers the best cup so far and explores variations around it. When would this directed exploration outperform independent sampling?

<details><summary>Thought</summary>

Basin hopping excels when good recipes cluster in a region of the parameter space. By jumping from the current best brew, it explores nearby basins more thoroughly. It struggles when the best coffee is isolated far from other good brews, because the jump range may not reach it. For our landscape, starting near x=5 (the best coffee) and jumping with a moderate range should reliably explore all three wells.
</details>

Implement `basin_hopping`. Starting from `x0`, repeat `n_jumps` times: (1) converge with Newton, (2) update the best if improved, (3) jump randomly from the best position. Clip the jump destination to `bounds`.

Return `(best_x, best_f, jump_history)` where `jump_history` records the initial x0 plus each jump destination.

Useful operations: `np.random.uniform(-jump_range, jump_range)`, `np.clip(x, low, high)`.

In [ ]:
def basin_hopping(newton_fn, f, f_prime, f_double_prime,
                  x0, n_jumps, jump_range, bounds):
    """Basin hopping: alternate between Newton convergence and random jumps."""
    # Step 1: initialise current_x = x0, best_x = x0, best_f = f(x0),
    #         jump_history = [x0]

    # Step 2: loop n_jumps times (track index i):
    #   - converge from current_x: call newton_fn -> x_local; evaluate f_local = f(x_local)
    #   - if f_local < best_f: update best_x, best_f
    #   - if this is not the last jump:
    #       delta = np.random.uniform(-jump_range, jump_range)
    #       current_x = float(np.clip(best_x + delta, bounds[0], bounds[1]))
    #       append current_x to jump_history

    # Step 3: return (best_x, best_f, jump_history)


check_basin_hopping(
    basin_hopping, newton_optimize,
    f, f_prime, f_double_prime,
    x0=0.0, n_jumps=8, jump_range=3.0, bounds=BOUNDS,
    known_global_x=5.0, known_global_f=-6.09,
)

### 📊 Method Comparison

> We now have three approaches to finding the best coffee: a single Newton run (one barista, one starting recipe), random restart (many baristas, each starting from a random recipe), and basin hopping (one barista who remembers the best cup and explores variations). Which strategy gives the best result per derivative evaluation?

<details><summary>Thought</summary>

A single Newton run is the cheapest but only finds one local brew. Random restart is embarrassingly parallel and guaranteed to explore broadly, but each barista works independently  -  it wastes effort re-tasting brews others have already found. Basin hopping reuses information (jump from the current best recipe) and can be more efficient in structured flavor landscapes, but its effectiveness depends on the jump range relative to the distance between sweet spots.
</details>

Run all three methods and display a comparison table. The cell below is provided  -  just run it after implementing the exercises above.

In [ ]:
# Single Newton from x0=0
x_newton, hist_newton, evals_newton = newton_optimize(f, f_prime, f_double_prime, x0=0.0)

# Random Restart (K=10)
np.random.seed(42)
rr_result = random_restart(newton_optimize, f, f_prime, f_double_prime,
                           n_restarts=10, bounds=BOUNDS)

# Basin Hopping (8 jumps from x0=0)
np.random.seed(42)
bh_result = basin_hopping(newton_optimize, f, f_prime, f_double_prime,
                          x0=0.0, n_jumps=8, jump_range=3.0, bounds=BOUNDS)

# Print comparison
print(f"{'Method':<30} {'x*':>10} {'f(x*)':>12}")
print(f"{'-'*54}")
print(f"{'Newton (x0=0)':<30} {x_newton:>10.6f} {f(x_newton):>12.6f}")

if rr_result is not None and isinstance(rr_result, tuple) and len(rr_result) >= 2:
    print(f"{'Random Restart (K=10)':<30} {rr_result[0]:>10.6f} {rr_result[1]:>12.6f}")
else:
    print(f"{'Random Restart (K=10)':<30} {'?':>10} {'?':>12}")

if bh_result is not None and isinstance(bh_result, tuple) and len(bh_result) >= 2:
    print(f"{'Basin Hopping (8 jumps)':<30} {bh_result[0]:>10.6f} {bh_result[1]:>12.6f}")
else:
    print(f"{'Basin Hopping (8 jumps)':<30} {'?':>10} {'?':>12}")

print()
print('Global minimum is near x=5 with f(5) \u2248 -6.09')

### 🏁 Recap

**What we did:**
- ☕ Explored a multi-modal coffee flavor landscape with three sweet spots of different quality.
- 🧭 Ran Newton from different starting recipes and saw that it always converges to the nearest local brew  -  not necessarily the best one.
- 🎲 Implemented random restart  -  brute-force global search by sending many baristas to random starting recipes.
- 🦘 Implemented basin hopping  -  a smarter strategy where the barista remembers the best cup and explores variations around it.
- 📊 Compared the three approaches on the same flavor landscape.

**Key takeaways:**
- Local optimizers like Newton find *a* good brew, not *the* best brew.
- Random restart is simple and reliable but does not reuse information between runs.
- Basin hopping can be more efficient by exploring nearby basins from a known good recipe.

**Now head back for self-check questions and key learnings in the lecture notes.**

## Take It from Here  -  Next Steps (Optional)

The exercises below are **optional** extensions. They deepen your intuition but are not required to follow the rest of the course. Work through them at your own pace after the session.

### 🗺️ Basin Mapping

> Every starting recipe in the domain belongs to exactly one basin of attraction  -  the local brew that Newton converges to from that point. If you colour each starting point by its destination, what does the boundary between basins look like, and does it always sit at a local maximum?

<details><summary>Thought</summary>

Basin boundaries coincide with unstable fixed points of the Newton map (typically local maxima or saddle points of f). At these points the gradient is zero but the curvature is negative, so Newton would try to step away. In 1D, the boundary between two basins is a single point; in higher dimensions it becomes a complex manifold.
</details>

The code below sweeps starting points across the domain, runs Newton from each, and colours the result by which brew it converged to.

In [ ]:
# Basin mapping
n_map = 500
x_starts = np.linspace(BOUNDS[0], BOUNDS[1], n_map)
converged_to = np.zeros(n_map)

for i, x0 in enumerate(x_starts):
    x_star, _, _ = newton_optimize(f, f_prime, f_double_prime, x0)
    converged_to[i] = x_star

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

# Top: landscape
xs_plot = np.linspace(BOUNDS[0], BOUNDS[1], 500)
ys_plot = np.array([f(xi) for xi in xs_plot])
ax1.plot(xs_plot, ys_plot, color=_GOLDEN, ls='-', linewidth=1.2)
ax1.set_ylabel('f(x)')
tufte_axis(ax1)

# Bottom: basin map
ax2.scatter(x_starts, converged_to, c=converged_to, cmap='viridis', s=3, linewidths=0)
ax2.set_xlabel('Starting point x0')
ax2.set_ylabel('Converged to x*')
tufte_axis(ax2)

plt.tight_layout()
plt.show()

### 🔧 The |f''| Trick

> When Newton encounters a region where f''(x) < 0 (concave curvature), it steps towards a maximum instead of a minimum. Replacing f''(x) with |f''(x)| in the update rule forces every step downhill. When does this help, and when might it cause problems?

<details><summary>Thought</summary>

The |f''| trick guarantees that the step direction is always towards decreasing f, which prevents convergence to maxima. However, it changes the step size and can cause the method to oscillate or converge more slowly near inflection points. It is most useful as a safeguard when the starting point is far from any minimum and the curvature sign is unpredictable.
</details>

Try running Newton with and without the |f''| trick on a simple concave function g(x) = -x^2 + 4x, which has a maximum at x=2 and no minimum.

In [ ]:
# Simple concave function: g(x) = -x^2 + 4x
g = lambda x: -x**2 + 4*x
g_prime = lambda x: -2*x + 4
g_double_prime = lambda x: -2.0

# Standard Newton  -  converges to the maximum at x=2
x_std, hist_std, _ = newton_optimize(g, g_prime, g_double_prime, x0=5.0)
print(f'Standard Newton:  x* = {x_std:.6f}  g(x*) = {g(x_std):.6f}')
print(f'  This is a MAXIMUM (f\"={g_double_prime(x_std):.1f} < 0)\n')

# Newton with |f''| trick
def newton_abs_fpp(f, f_prime, f_double_prime, x0, tol=1e-8, max_iter=100):
    x = x0
    history = [x]
    for i in range(max_iter):
        fpx = f_prime(x)
        fppx = abs(f_double_prime(x))
        if fppx < 1e-15:
            break
        x_new = x - fpx / fppx
        history.append(x_new)
        if abs(x_new - x) < tol:
            break
        x = x_new
    return history[-1], history, len(history)

x_abs, hist_abs, _ = newton_abs_fpp(g, g_prime, g_double_prime, x0=5.0)
print(f'|f\'\' | Newton:  x* = {x_abs:.6f}  g(x*) = {g(x_abs):.6f}')
print(f'  The method diverges to -inf because g has no minimum.')
print(f'  Every step moves downhill, but there is no bottom to reach.')